# NB05 — Third Backbone: Is the Coupling Architecture-Specific?

**Established (NB02-04):** competence-calibration coupling is robust for **Xception**
(r=-0.80 to -0.94, confound-broken, competence-driven).

**This notebook:** does it hold for a DIFFERENT architecture? Run **EfficientNetB4**
(weights/train_on_fs/efficientnetb4.pth) through the same pipeline on FF++ + a few DF40 methods.

**Decision:**
- EfficientNetB4 shows the same coupling -> **architecture-general finding** (strong) ✓
- It doesn't -> coupling is Xception-specific (narrows the claim)

Reuses everything. inference.py already supports 'efficientnetb4' in _DETECTOR_SPECS.
NOTE: EfficientNetB4 config differs from Xception — check the efficientnetb4.yaml for
resolution/normalization before scoring (next cell verifies).


## Cell 1 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, sys, glob, subprocess
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
PARENT = "/content/drive/MyDrive/CDTS_Research"
DFB = f"{REPO}/external/DeepfakeBench"
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"{PARENT}/{f}"): subprocess.run(f'cp "{PARENT}/{f}" /root/{f}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
print("installing deps..."); subprocess.run("pip -q install efficientnet_pytorch timm einops kornia simplejson", shell=True)
print("EfficientNetB4 ckpt:", os.path.exists(f"{REPO}/weights/train_on_fs/efficientnetb4.pth"))
print("FF++ manifest:", os.path.exists(f"{REPO}/data/ffpp_manifest.parquet"))

Mounted at /content/drive
installing deps...
EfficientNetB4 ckpt: True
FF++ manifest: True


## Cell 2 — Check EfficientNetB4 config (preprocessing contract may differ from Xception)

Xception used 256px, mean/std 0.5. EfficientNetB4 might differ. Read its yaml before scoring —
a wrong preprocessing contract gives garbage (the 0.79-not-0.97 lesson from NB02).


In [2]:
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
import yaml
cfg = yaml.safe_load(open(f"{REPO}/external/DeepfakeBench/training/config/detector/efficientnetb4.yaml"))
print("resolution:", cfg.get("resolution"))
print("mean:", cfg.get("mean"), "std:", cfg.get("std"))
print("backbone_name:", cfg.get("backbone_name"))
print("backbone_config:", cfg.get("backbone_config"))

resolution: 256
mean: [0.5, 0.5, 0.5] std: [0.5, 0.5, 0.5]
backbone_name: efficientnetb4
backbone_config: {'num_classes': 2, 'inc': 3, 'dropout': False, 'mode': 'Original'}


## Cell 3 — Load EfficientNetB4 + score FF++

If the config shows non-default resolution/normalization, we'd need to pass it to
score_manifest. inference.py uses 256 + mean/std 0.5 by default — if EfficientNetB4 matches,
fine; if not, flag it and we adjust. (EfficientNetB4 in DeepfakeBench commonly uses 256 too.)


In [ ]:
import sys, importlib.util, torch, hashlib, os
import pandas as pd
from sklearn.metrics import roc_auc_score
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB = f"{REPO}/external/DeepfakeBench"

for k in list(sys.modules.keys()):
    if k.startswith("detectors") or k.startswith("networks") or k=="metrics" or k.startswith("metrics.") or k=="inference":
        del sys.modules[k]
sys.path = [p for p in sys.path if p not in (f"{DFB}/training", f"{REPO}/src", DFB)]
sys.path.insert(0, DFB); sys.path.insert(0, f"{DFB}/training"); sys.path.append(f"{REPO}/src")

spec = importlib.util.spec_from_file_location("inference", f"{REPO}/src/inference.py")
inference = importlib.util.module_from_spec(spec); sys.modules["inference"]=inference
spec.loader.exec_module(inference)

ckpt = f"{REPO}/weights/train_on_fs/efficientnetb4.pth"
model, device, info = inference.load_detector(dfb_root=DFB, backbone_name="efficientnetb4", ckpt_path=ckpt)
print("EfficientNetB4 load:", info, "device:", device)

# score FF++ test
mani = pd.read_parquet(f"{REPO}/data/ffpp_manifest.parquet")
scores = inference.score_manifest(model, device, mani, splits=["test"], batch_size=64, verbose=False)
scores.to_parquet(f"{REPO}/reports/scores/effnetb4_ffpp_test.parquet", index=False)

# per-method AUC — SANITY: faceswap should be high (~0.9+) if preprocessing is right
reals = scores[scores.label==0]
print("\n=== EfficientNetB4 FF++ AUC per method ===")
for m in ['faceswap','deepfakes','face2face','neuraltextures']:
    fk = scores[(scores.label==1)&(scores.method==m)]
    sub = pd.concat([reals, fk])
    print(f"  {m:16s}: AUC {roc_auc_score(sub.label, sub.prob_fake):.4f}")
print("\n(if faceswap < 0.85, preprocessing contract may be wrong — check Cell 2 config)")

EfficientNetB4 load: {'missing': 0, 'unexpected': 0} device: cuda


## Cell 4 — Calibrate EfficientNetB4 FF++ + coupling (per-method)

In [ ]:
import sys, importlib.util, os
import numpy as np, pandas as pd
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

for k in list(sys.modules.keys()):
    if k=="metrics" or k.startswith("metrics.") or k=="calibration":
        del sys.modules[k]
sys.path = [p for p in sys.path if "DeepfakeBench" not in p]
if f"{REPO}/src" in sys.path: sys.path.remove(f"{REPO}/src")
sys.path.insert(0, f"{REPO}/src")
import calibration as cal, metrics as met

scores = pd.read_parquet(f"{REPO}/reports/scores/effnetb4_ffpp_test.parquet")
rows = []
reals = scores[scores.label==0]
for m in ['faceswap','deepfakes','face2face','neuraltextures']:
    sub = pd.concat([reals, scores[(scores.label==1)&(scores.method==m)]]).reset_index(drop=True)
    p = sub.prob_fake.values.astype(float); y = sub.label.values.astype(int)
    g = sub.identity_id.values if "identity_id" in sub.columns else None
    ci, ti, _ = cal.leakage_safe_split(y, groups=g, calib_frac=0.5, seed=42)
    p_cal, _ = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)
    auc = met.roc_auc(p[ti], y[ti]) if hasattr(met,"roc_auc") else float("nan")
    ece_cal = met.ece(p_cal, y[ti], n_bins=15, scheme="equal_mass")
    rows.append({"method":f"effnetb4_{m}","backbone":"efficientnetb4","AUC":auc,"ECE_cal":ece_cal})
    print(f"  {m:16s}  AUC={auc:.4f}  ECE_cal={ece_cal:.4f}")

eff = pd.DataFrame(rows)
eff.to_csv(f"{REPO}/reports/calibration/coupling_effnetb4_ffpp.csv", index=False)

from scipy.stats import pearsonr
if eff.AUC.nunique() > 2:
    r,p = pearsonr(eff.AUC, eff.ECE_cal)
    print(f"\nEfficientNetB4 FF++ coupling (4 pts): r={r:.3f} (p={p:.4f})")
print("\n=> compare to Xception's coupling. Same direction = architecture-general.")

## Cell 5 — Combined: does the coupling hold across BOTH architectures?

Pool EfficientNetB4 points with the existing Xception 20-point set. If the combined
correlation stays strong and EfficientNetB4 points fall on the same line, the coupling is
architecture-general — a much stronger claim.


In [ ]:
import pandas as pd, numpy as np, os, glob
from scipy.stats import pearsonr, spearmanr
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

xcep = pd.read_csv(f"{REPO}/reports/calibration/coupling_all_16pts.csv")[["AUC","ECE_cal"]]; xcep["backbone"]="xception"
frm = pd.read_csv(f"{REPO}/reports/calibration/coupling_df40_FRmismatch.csv")[["AUC","ECE_cal"]]; frm["backbone"]="xception"
eff = pd.read_csv(f"{REPO}/reports/calibration/coupling_effnetb4_ffpp.csv")[["AUC","ECE_cal"]]; eff["backbone"]="efficientnetb4"
allp = pd.concat([xcep, frm, eff]).reset_index(drop=True)

r,p = pearsonr(allp.AUC, allp.ECE_cal); rs,ps = spearmanr(allp.AUC, allp.ECE_cal)
print(f"ALL backbones ({len(allp)} pts): Pearson r={r:.3f} (p={p:.2e}) | Spearman rho={rs:.3f}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.5,5.5))
cmap = {"xception":"#2471A3","efficientnetb4":"#E67E22"}
for bb,g in allp.groupby("backbone"):
    ax.scatter(g.AUC, g.ECE_cal, s=90, alpha=0.85, label=f"{bb} (n={len(g)})",
               color=cmap.get(bb,"gray"), edgecolor="white", linewidth=1.2, zorder=3)
x,y=allp.AUC.values, allp.ECE_cal.values
b,a=np.polyfit(x,y,1); xs=np.linspace(x.min()-.02,x.max()+.02,100)
ax.plot(xs,a+b*xs,"k--",lw=1.8,label="OLS (all)")
ax.annotate(f"r={r:.2f} (p={p:.1e})",xy=(.97,.95),xycoords="axes fraction",ha="right",va="top",
            fontsize=11,bbox=dict(boxstyle="round,pad=0.4",fc="white",ec="gray"))
ax.set_xlabel("Detection competence (AUC)",fontsize=12); ax.set_ylabel("ECE$_{cal}$",fontsize=12)
ax.set_title("Competence-Calibration Coupling across architectures",fontsize=12)
ax.legend(loc="upper right",fontsize=9); ax.grid(True,alpha=0.25); plt.tight_layout()
out=f"{REPO}/figures/coupling_two_backbones.png"
plt.savefig(out,dpi=300,bbox_inches="tight"); plt.show()
print("saved:", out)

## Cell 6 — Commit NB05 results

In [ ]:
import os, subprocess
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
os.chdir(REPO)
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"/content/drive/MyDrive/CDTS_Research/{f}"):
        subprocess.run(f'cp "/content/drive/MyDrive/CDTS_Research/{f}" /root/{f}', shell=True)
subprocess.run("git add reports/scores/effnetb4_ffpp_test.parquet reports/calibration/coupling_effnetb4_ffpp.csv figures/coupling_two_backbones.png notebooks/NB05_third_backbone.ipynb", shell=True)
r = subprocess.run("git status --short", shell=True, capture_output=True, text=True)
print(r.stdout)
print(">>> review staged, then commit + push in next cell")